# Обучение Tri-View 3D Autoencoder (Этап 1)

Архитектура с skip connections от lifted views + улучшенный training pipeline.

**Запуск:** Google Colab с GPU T4.

In [ ]:
# 1. Монтируем Google Drive (для данных)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Скачиваем актуальный код с GitHub
%cd /content
!rm -rf TriView3D
!git clone https://github.com/3x6dll9ff/TriView3D.git
%cd TriView3D

In [ ]:
# 3. Устанавливаем зависимости
!pip install -r requirements.txt

In [ ]:
# 4. Распаковываем подготовленные данные
!rm -rf /content/data
!unzip -q /content/drive/MyDrive/data_processed.zip -d /content/
print("Распаковка завершена.")

import os, glob
matches = glob.glob('/content/**/metadata.csv', recursive=True)
DATA_PATH = os.path.dirname(matches[0]) if matches else ''
print(f"Данные: {DATA_PATH}")
print(f"Сэмплов: {len([f for f in os.listdir(os.path.join(DATA_PATH, 'top_proj')) if f.endswith('.npy')])}")

In [ ]:
# 5. Обучение base autoencoder
!DATA_PATH=$(find /content -name "metadata.csv" | head -n 1 | xargs dirname) && \
 echo "Используем данные из: $DATA_PATH" && \
 python3 src/train_reconstruction.py \
    --data_dir "$DATA_PATH" \
    --input_mode tri \
    --epochs 80 \
    --batch_size 8 \
    --lr 1e-3 \
    --num_workers 2 \
    --early_stopping_patience 15

In [ ]:
# 6. Скачиваем модель на компьютер
from google.colab import files
import os
if os.path.exists("results/best_autoencoder.pt"):
    files.download("results/best_autoencoder.pt")
else:
    print("ОШИБКА: best_autoencoder.pt не найден!")